### Reading Log Results

In [1]:
import os
import json
import gzip
import numpy as np
from tqdm import tqdm
from collections import defaultdict

In [ ]:
# ---- Utils ------

dist_btw_pts = lambda pt_1, pt_2: np.linalg.norm(pt_1 - pt_2)

def get_obj_pos(scene_name: str, obj_cat: str, data_info_dir: str):
    """"
    Returns lists of all object instance positions and viewpoints

    Args:
        - scene_name
        - obj_cat: Object Category name Ex.: table cloth
        - data_info_dir: Dataset content directory
    """

    #Positions of multiple instances of the object category
    obj_pos = []     
    obj_view_pts = []   

    #Key for the goal in the scene
    goal_key = f"{scene_name}.basis.glb_{obj_cat}"

    scene_info_path = os.path.join(data_info_dir, f"{scene_name}.json.gz")
    with gzip.open(scene_info_path, "r") as f:
        scene_info = json.load(f)
    
    #Iterate through each goal instance, and save the position info
    for goal_instance in scene_info["goals_by_category"][goal_key]:

        if goal_instance["object_category"] != obj_cat: continue

        curr_view_pts = goal_instance["view_points"]
        curr_view_pts = [pt["agent_state"]["position"] for pt in curr_view_pts]                                      

        obj_pos.append( goal_instance["position"] )
        obj_view_pts.append( curr_view_pts )

    return np.array(obj_pos), np.vstack(obj_view_pts)

def calculate_dist_to_goal(scene_id, episode_id,
                            target_obj_descr, traj_dir, data_content_dir,
                            data_mode = None):
    """
    Returns distance to goal position and distance to nearest view point
    """

    traj_path = os.path.join(traj_dir, f"{str(episode_id)}_{scene_id}.txt")
    assert os.path.exists(traj_path), f"Trajectory file {traj_path} does not exist."
    
    #Get path length from trajectory
    with open(traj_path, "r") as f:
        traj = f.readlines()

    traj = np.array([[float(elem) for elem in row.strip().split(",")] for row in traj])[:-1]
    final_pos = traj[-1, [1, 3]]

    #Get target position
    target_pos, target_vw_pts = get_obj_pos(scene_id, target_obj_descr, data_content_dir)
    target_pos, target_vw_pts = target_pos[:, [0, 2]], target_vw_pts[:, [0, 2]]

    # print(f"\nFinal : {final_pos},\nTarget: {target_pos}\n")

    view_pt_dists = np.apply_along_axis(lambda v: dist_btw_pts(v, final_pos), axis=1, arr=target_vw_pts)
    dist_to_closest_view_pt = min(view_pt_dists)

    obj_dists = np.apply_along_axis(lambda v: dist_btw_pts(v, final_pos), axis=1, arr=target_pos)
    dtg_curr = min(obj_dists)

    return dtg_curr, dist_to_closest_view_pt

def get_spl_traj(scene_id, episode_id, 
                 target_obj_descr, traj_dir, data_content_dir, 
                 data_mode=None):

    traj_path = os.path.join(traj_dir, f"{str(episode_id)}_{scene_id}.txt")
    # if not os.path.exists(traj_path): return 0
    assert os.path.exists(traj_path), f"Trajectory file {traj_path} does not exist."
    
    #Get path length from trajectory
    with open(traj_path, "r") as f:
        traj = f.readlines()

    traj = np.array([[float(elem) for elem in row.strip().split(",")] for row in traj])[:-1]
    traj = traj[:, [1, 3]]
    
    deltas = traj[1:, :] - traj[:-1, :]
    traj_dist = np.linalg.norm(deltas, axis=1).sum()

    #Get optimal (GT) path length
    scene_info_path = os.path.join(data_content_dir, f"{scene_id}.json.gz")

    with gzip.open(scene_info_path, "r") as f:
        scene_info = json.load(f)

    geo_dist = None
    for ep in scene_info["episodes"]:

        ep_obj = ep["description"][0] if data_mode == "PersONAL" else ep["object_category"]

        if target_obj_descr == ep_obj:
            geo_dist = ep["info"]["geodesic_distance"]
            break

    if geo_dist is None: return None
    
    spl = geo_dist / max(geo_dist, traj_dist)
    return spl


# ----- Read Results -----

def read_logs(log_dir):
    results = {}
    invalids = {}

    file_counts = 0

    for file_name in tqdm(os.listdir(log_dir)):
        try:
            if "json" not in file_name: continue

            file_path = os.path.join(log_dir, file_name)
            with open(file_path, "r") as f:
                info = json.load(f)

            info = defaultdict(list, info)

            episode_id = file_name.split("_")[0]
            scene_id = "_".join( file_name.split(".json")[0].split("_")[1:] )

            results[(scene_id, episode_id)] = {
                "dist_to_goal": info["distance_to_goal"],
                "success": info["success"],
                "spl": info["spl"],
                "target_object": info["target_object"],
                "stop_called": info["stop_called"],
                "final_pos": info["final_pos"],
                "num_steps": info["num_steps"],
                "failure_case": info["failure_cause"]
            }

            file_counts += 1

        except Exception as e:
            print(f"Skipping file {file_name} due to Error {e}. Episode Success: {info['success']}")
            invalids[id] = info
            continue

    print(f"Loaded {file_counts}/{file_counts+len(invalids)} files")

    return results, invalids

def read_results(results, dist_thresh=0.1, 
                 with_stop=True, 
                 traj_dir=None, data_content_dir=None,
                 data_mode = None, dtg_mode="hab",
                 till_num = -1):
    """"
    Args:
        - results: Output of read_logs
        - dist_thresh: Distance Threshold to determine success
        - with_stop: Called stop is a necessary condition for success
        - traj_dir: Directory to all episodes' trajectories. Used for calculating SPL.
        - data_content_dir: Directory to scene content info. Used for calculating SPL, DTG.
        - data_mode: PersONAL or not
        - dtg_mode: Success criteria is determined using
                        1. hab : Habitat provided distance to goal
                        2. goal : Manually calculate distance to goal
                        3. view_pt : Manually calculate distance to closest view point
        - till_num: Filter the episodes till this upper limit
    """

    assert dtg_mode in ["hab", "goal", "view_pt"], "Please choose a valid dtg_mode arg"

    success = 0
    spl = 0

    keys = []

    spl_ratios = []
    spl_traj = []
    dtg = []

    num_eps = 0

    for k in results:
        scene_id, episode_id = k[0], k[1]

        #Extract relevant info from results
        steps = results[k]["num_steps"]
        stop_called = results[k]["stop_called"]
        spl_curr = results[k]["spl"]
        spl_curr = 0 if type(spl) is not int else spl_curr

        #Set dtg_valid for success thresholding
        #Also calculate distance to goal manually or from habitat depending on the mode
        if dtg_mode in ["goal", "view_pt"]:
            target_obj = results[k]["target_object"]
            dist_to_goal, dist_to_goal_vw = calculate_dist_to_goal(scene_id, episode_id,
                                                                    target_obj, traj_dir, 
                                                                    data_content_dir)

            if dtg_mode == "goal":
                dtg_valid = dist_to_goal
            else:
                dtg_valid = dist_to_goal_vw
        else:
            dist_to_goal = results[k]["dist_to_goal"]
            dtg_valid = dist_to_goal


        #Success Thresholding
        #Condition for including called stop as a requisite for success
        #If with_stop is False, the only criteria for success is distance to target
        if (not with_stop) and (steps > 498): stop_called = True

        if (dtg_valid < dist_thresh) and (stop_called):
            success += 1

            keys.append(k)
            spl_ratios.append(spl_curr)

            #Calculate SPL from trajectory
            if (traj_dir) and (data_content_dir):
                spl_traj.append(
                    get_spl_traj(scene_id, episode_id, 
                                 target_obj_descr = results[k]['target_object'], 
                                 traj_dir = traj_dir,
                                 data_content_dir = data_content_dir,
                                 data_mode = data_mode
                                 )
                )


        dtg.append(dist_to_goal)

        if (till_num > 0) and (num_eps > till_num):
            break

        num_eps += 1
    
    
    dtg = [elem for elem in dtg if not np.isinf(elem)]

    # num_eps = len(results)
    print(f"Success Rate : {success}/{num_eps} -> {(success/num_eps) * 100}")
    print(f"SPL : {(sum(spl_ratios)/num_eps)}")
    print(f"DTG : {np.mean(dtg)}")

    if len(spl_traj) > 0:
        print(f"\nSPL (Traj): {sum(spl_traj)/num_eps}")

    return keys



In [15]:
#WACV Rebuttal : OVON Equal, VLFM Stand
data_content_dir = "habitat-lab/data/datasets/ovon/hm3d/val_seen_synonyms_equal/content"
log_dir = "/mnt/vlfm_stand/logs/WACV_Rebuttal/ovon_equal/vlfm_stand"
traj_dir = os.path.join(log_dir, 'trajectory')

#Check results: OVON Equal, VLFM Stand, Updated Hidden state shape
# log_dir = "/mnt/vlfm_stand/logs/WACV_Rebuttal/ovon_equal/vlfm_hidden_check"
# traj_dir = os.path.join(log_dir, 'trajectory')


results, errors = read_logs(log_dir)

100%|██████████| 602/602 [00:00<00:00, 31312.42it/s]

Loaded 600/600 files


In [16]:
success_keys = read_results(results, 
                    dist_thresh = 0.2,
                    with_stop = False,
                    traj_dir = traj_dir,
                    data_content_dir=data_content_dir,
                    data_mode = None,
                    )

Success Rate : 40/600 -> 6.666666666666667
SPL : 0.0003869109367268586
DTG : 5.753246171083301

SPL (Traj): 0.0188193477635749


### Dataset Meddlings

#### Reading Dataset Content

In [4]:
import os
import gzip
import json

In [18]:
data_content_dir = "habitat-lab/data/datasets/ovon/hm3d/val_seen_synonyms_equal/content"

scene_id = "TEEsavR23oF"
ep_id = "5127"
target_obj = "armchair" #"bath sink"

scene_info_path = os.path.join(data_content_dir, f"{scene_id}.json.gz")
with gzip.open(scene_info_path, "r") as f:
    scene_info = json.load(f)

scene_info.keys()

dict_keys(['goals_by_category', 'episodes', 'category_to_task_category_id', 'category_to_scene_annotation_category_id'])

In [12]:
if target_obj is not None:

    target_key = f"{scene_id}.basis.glb_{target_obj}"

    for goal_instance in scene_info["goals_by_category"][target_key]:
        print(goal_instance)

{'object_category': 'armchair', 'object_id': 'armchair_30', 'position': [1.44466, 3.56965, -5.65514], 'view_points': [{'agent_state': {'position': [0.55083, 3.11338, -5.58263], 'rotation': [0.0, -0.67792, 0.0, 0.73514]}, 'iou': 0.59195}, {'agent_state': {'position': [0.30083, 3.11338, -5.58263], 'rotation': [0.0, -0.68437, 0.0, 0.72913]}, 'iou': 0.4889}, {'agent_state': {'position': [0.30083, 3.11338, -5.83263], 'rotation': [0.0, -0.75939, 0.0, 0.65064]}, 'iou': 0.47413}, {'agent_state': {'position': [0.30083, 3.11437, -6.08263], 'rotation': [0.0, -0.82161, 0.0, 0.57005]}, 'iou': 0.44727}, {'agent_state': {'position': [0.30083, 3.1182, -6.33263], 'rotation': [0.0, -0.8688, 0.0, 0.49517]}, 'iou': 0.41169}, {'agent_state': {'position': [0.05083, 3.11338, -5.58263], 'rotation': [0.0, -0.68849, 0.0, 0.72524]}, 'iou': 0.4043}, {'agent_state': {'position': [0.05083, 3.11338, -5.83263], 'rotation': [0.0, -0.75044, 0.0, 0.66094]}, 'iou': 0.38937}, {'agent_state': {'position': [0.05083, 3.12283

In [7]:
scene_info["goals_by_category"]

{'TEEsavR23oF.basis.glb_armchair': [{'object_category': 'armchair',
   'object_id': 'armchair_30',
   'position': [1.44466, 3.56965, -5.65514],
   'view_points': [{'agent_state': {'position': [0.55083, 3.11338, -5.58263],
      'rotation': [0.0, -0.67792, 0.0, 0.73514]},
     'iou': 0.59195},
    {'agent_state': {'position': [0.30083, 3.11338, -5.58263],
      'rotation': [0.0, -0.68437, 0.0, 0.72913]},
     'iou': 0.4889},
    {'agent_state': {'position': [0.30083, 3.11338, -5.83263],
      'rotation': [0.0, -0.75939, 0.0, 0.65064]},
     'iou': 0.47413},
    {'agent_state': {'position': [0.30083, 3.11437, -6.08263],
      'rotation': [0.0, -0.82161, 0.0, 0.57005]},
     'iou': 0.44727},
    {'agent_state': {'position': [0.30083, 3.1182, -6.33263],
      'rotation': [0.0, -0.8688, 0.0, 0.49517]},
     'iou': 0.41169},
    {'agent_state': {'position': [0.05083, 3.11338, -5.58263],
      'rotation': [0.0, -0.68849, 0.0, 0.72524]},
     'iou': 0.4043},
    {'agent_state': {'position': [0

In [19]:
for ep in scene_info["episodes"]:

    if ep["episode_id"] == ep_id:
        print('Found!')
        break


ep

Found!


{'episode_id': '5127',
 'scene_id': 'hm3d/val//00800-TEEsavR23oF/TEEsavR23oF.basis.glb',
 'scene_dataset_config': './data/scene_datasets/hm3d/hm3d_annotated_basis.scene_dataset_config.json',
 'additional_obj_config_paths': [],
 'start_position': [-11.13075, 0.01338, -2.1485],
 'start_rotation': [0, 0.25704, 0, -0.9664],
 'info': {'geodesic_distance': 6.297317028045654,
  'euclidean_distance': 5.849415431101938},
 'goals': [],
 'start_room': None,
 'shortest_paths': None,
 'object_category': 'bath sink',
 'children_object_categories': []}

#### OVON : Equal Representation of Categories

In [ ]:
import gzip
import json
import os
import numpy as np
from tqdm import tqdm

In [ ]:
ovon_dir = "/mnt/vlfm_query_embed/habitat-lab/data/datasets/ovon/hm3d/val_seen_synonyms/content"
ovon_dir = "/mnt/vlfm_query_embed/habitat-lab/data/datasets/ovon/hm3d/val_seen/content"
# ovon_dir = "/mnt/vlfm_query_embed/habitat-lab/data/datasets/ovon/hm3d/val_unseen/content"

valid_scenes = [k.replace('.json.gz', '') for k in os.listdir(ovon_dir) if k.__contains__("json.gz") and not k.startswith('.')]
print(valid_scenes)

In [ ]:
# ----- Utils ------

def map_cat_to_eps(ovon_content_dir):
    """"
    Creates a mapping (dict) of unique category name to list 
    of (scene, episode) pairs containing the category
    """

    cat_to_eps = {}
    num_eps = 0

    for file in tqdm(os.listdir(ovon_content_dir)):

        if ("json.gz" not in file) or (file.startswith(".")): continue

        scene_name = file.split(".json")[0] 

        file_path = os.path.join(ovon_content_dir, file)
        with gzip.open(file_path, "r") as f:
            content = json.load(f)

        for ep in content['episodes']:
            cat_name = ep['object_category']
            ep_id = ep["episode_id"]

            if cat_name in cat_to_eps:
                cat_to_eps[cat_name].append((scene_name, ep_id))
            else:
                cat_to_eps[cat_name] = [(scene_name, ep_id)]

        num_eps += len(content['episodes'])

    print(f"Extracted {len(cat_to_eps)} unique categories from {num_eps} episodes.")
    return cat_to_pos

def sample_equal_eps(cat_to_eps, num_samples=600):
    """
    Samples (scene, episode) pairs to approximate an 
    uniform distribution over all unique categories.

    Three stages:
        - Initial : Based on number of cats, assign equal number of samples (at most) for each category.
                    Since some categories might have a lower number of samples than is required for a perfect equal division,
                    we try to redistribute the remaining extra pairs.
        - Recursive: If it is possible to redistribute the extra pairs equally among the remaining categories. If not, we go to the final stage.
        - Final: Randomly sample a category and assign a sampled episode from its extra samples, and keep doing this till the num_samples limit is hit.
    """

    sampled_eps = {}
    extra_eps = {}
    num_uniq_cats = len(cat_to_eps)
    count_samples = 0

    #Assigns equal number of samples to each category, while also tracking categories with extra samples
    # Here, a sample refers to a (scene, episode) pair
    print(f" - Sampling : Initial Stage")
    least_cat_samples = num_samples//num_uniq_cats
    for cat in cat_to_eps:
        eps = cat_to_eps[cat]
        if len(eps) <= least_cat_samples:
            sampled_eps[cat] = eps
        else:
            sample_args = np.random.choice(len(eps), least_cat_samples, replace=False)
            sampled_eps[cat] = [eps[i] for i in sample_args]
            extra_eps[cat] = list( set(eps) - set(sampled_eps[cat]) )

        count_samples += len(sampled_eps[cat])

    #If there are remaining samples, use the extra_eps to recursively fill them equally across categories
    remaining_samples = num_samples - count_samples
    print(f" - Sampling : Remaining Samples: {remaining_samples}")

    if (remaining_samples > 0) and (remaining_samples//len(extra_eps)) > 0:
        print(f" - Sampling : Recursive Stage")
        extra_sampled_eps, extra_eps = sample_equal_eps(extra_eps, remaining_samples)
        for cat in extra_sampled_eps:
                sampled_eps[cat].extend(extra_sampled_eps[cat])
    else:
        print("Sampling : Final Stage")
        extra_cats = list(extra_eps.keys())
        np.random.shuffle(extra_cats)

        for cat in extra_cats:
            if remaining_samples == 0: break

            eps = extra_eps[cat]
            ep_sample_arg = np.random.choice(len(eps), 1)
            sampled_eps[cat].extend([eps[ep_sample_arg[0]]])
            remaining_samples -= 
            # print(cat)
    
    return sampled_eps, extra_eps

def conv_key_cat_to_scene(sampled_eps):
    """
    Convert the category to list of (scene, episode) pairs mapping to 
    a mapping from scene to list of episodes.
    """

    scene_eps = {}
    for cat in sampled_eps:
        for (scene, ep) in sampled_eps[cat]:
            
            if scene in scene_eps:
                scene_eps[scene].append(ep)
            else:
                scene_eps[scene] = [ep]

    return scene_eps

In [ ]:
cat_to_eps = map_cat_to_eps(ovon_dir)
sampled_eps, _ = sample_equal_eps(cat_to_eps, num_samples=600)
equal_scene_eps = conv_key_cat_to_scene(sampled_eps)

#Sanity check
len_equal_scene_eps = {k : len(v) for k, v in equal_scene_eps.items()}
sum(len_equal_scene_eps.values())

In [ ]:
#Copy and move the valid (scene, episode) pairs

ovon_new_dir = "/mnt/vlfm_query_embed/habitat-lab/data/datasets/ovon/hm3d/val_seen_synonyms_equal/"
ovon_new_content_dir = os.path.join(ovon_new_dir, "content")
# os.makedirs(ovon_new_content_dir, exist_ok=True)

#After copying the old dir manually and renaming, we can now filter the content
for file in tqdm(os.listdir(ovon_new_content_dir)):

    if "json.gz" not in file: continue

    scene_name = file.split(".json")[0]
    if scene_name not in equal_scene_eps.keys(): continue

    file_path = os.path.join(ovon_dir, file)
    with gzip.open(file_path, "r") as f:
        content = json.load(f)

    filtered_eps = [ep for ep in content['episodes'] if ep['episode_id'] in equal_scene_eps[scene_name]]
    content['episodes'] = filtered_eps

    new_file_path = os.path.join(ovon_new_content_dir, file)
    # with gzip.open(new_file_path, "wt") as f:
    #     json.dump(content, f)

In [ ]:
#Sanity Check
total_eps = 0
ovon_new_eps = {}
for file in tqdm(os.listdir(ovon_new_content_dir)):

    if "json.gz" not in file: continue

    scene_name = file.split(".json")[0]
    if scene_name not in equal_scene_eps.keys(): continue

    file_path = os.path.join(ovon_new_content_dir, file)
    with gzip.open(file_path, "r") as f:
        content = json.load(f)

    total_eps += len(content['episodes'])
    ovon_new_eps[scene_name] = len(content['episodes'])

### Saving Results

#### Saving to MP4 Video

In [ ]:
import cv2
import glob
import os
from tqdm import tqdm

def save_to_mp4(imgs_dir, save_name, fps=15, save_dir=None):

    assert ".mp4" in save_name, "save_name should contain the .mp4 extension"
    save_path = os.path.join(os.path.dirname(imgs_dir), save_name)

    # Get list of image files (sorted by name)
    img_files = sorted(glob.glob(f"{imgs_dir}/*.png"))  # or .jpg
    assert len(img_files) > 0, f"No images found in directory : {imgs_dir}"

    # Read first image to get size
    frame = cv2.imread(img_files[0])
    h, w, _ = frame.shape

    # Define video writer (MP4, 30 fps)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # use 'XVID' for .avi
    out = cv2.VideoWriter(save_path, fourcc, fps, (w, h))

    for fname in tqdm(img_files):
        img = cv2.imread(fname)
        out.write(img)

    out.release()
    print(f"Video saved to : {save_path}")

In [ ]:
# root_dir = "/mnt/vlfm_query_embed/embed_results/WACV_Rebuttal/ovon_equal/vlfm_imprint_fails/stop_feature_map/gifs"

# for folder_name in tqdm(os.listdir(root_dir)):

#     folder_path = os.path.join(root_dir, folder_name)
#     save_to_mp4(folder_path, f"{folder_name}.mp4")